In [ ]:
#Instalação e importação das bibliotecas necessárias
!pip install prefect
!pip install loguru
!pip install pycountry
import pandas as pd
import numpy as np
from prefect import task, flow
import pycountry

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 55.1 MB/s eta 0:00:00


In [ ]:
#Configurar loguru
from loguru import logger
import sys
import logging

#Ocultar mensagens do Prefect que não sejam warning
logging.getLogger("prefect").setLevel(logging.WARNING)

logger.remove()

logger.add(
    sys.stdout,
    format="<green>{time:HH:mm:ss}</green> | "
           "<level>{level:^9}</level> | "
           "<cyan>{function}</cyan>: "
           "{message}",
    level="INFO",
    colorize=True
)

#Cria arquivo de log que não excederá 1mb
logger.add(
    "pipeline.log",
    rotation="1 MB",
    level="INFO",
    encoding="utf-8"
)

2

In [ ]:
#Visualização do dataset enriquecido

@task()
def criacao_df():
  logger.info("Iniciando extração...")

  df = pd.read_csv('df_netflix_enriquecido.csv')

  logger.info(f"Arquivo carregado com sucesso. Linhas: {len(df)}")
  return df

In [ ]:
#Remoção de duplicatas
def remover_duplicatas(df):
      duplicadas = df.duplicated().sum()
      logger.info(f"Iniciando verificação de duplicatas. Registros duplicados encontrados: {duplicadas}")
      df.drop_duplicates(inplace=True)
      logger.info("Duplicatas removidas!")
      return df

In [ ]:
#Remoção da coluna de descrição
def remover_descricao(df):
    logger.info("Iniciando remoção da coluna 'description'")
    df.drop(columns=['description'], inplace=True)
    logger.info("Remoção concluída!")
    return df

In [ ]:
#Substituição de Unknown e nulos para 'Não informado'

@task
def substituicao_nulos(df):
  nulos_inicio = df.isna().sum().sum()
  total_campos = df.size
  logger.info(f"Iniciando substituição de nulos. Campos nulos encontrados: {nulos_inicio}")

  df = df.replace(['Unknown', 'nao_informado'], 'Não Informado').fillna('Não Informado')

  nulos_final = df.isna().sum().sum()
  logger.info(f"Transformação concluída! Nulos: {nulos_final}")

 #Alerta muitos nulos:
  percentual_nulos = (nulos_final / total_campos) * 100
  if percentual_nulos > 10:
      logger.warning(f"Alto volume de valores nulos: {percentual_nulos:.2f}% do dataset.")

  return df

In [ ]:
#Enriquecer com colunas da API (director_api	cast_api	country_api)

@task
def enriquecer_com_api(df):
  logger.info("Iniciando merge com API...")

  df['cast'] = np.where(
      (df['cast'] == "Não Informado") &
      (df['cast_api']) != "Não Informado",
      df['cast_api'],
      df['cast']
  )

  df['director'] = np.where(
      (df['director'] == "Não Informado") &
      (df['director_api']) != "Não Informado",
      df['director_api'],
      df['director']
  )

  df['country'] = np.where(
      (df['country'] == "Não Informado") &
      (df['country_api']) != "Não Informado",
      df['country_api'],
      df['country']
  )

  #Dropar colunas da api
  df.drop(columns=['director_api', 'cast_api', 'country_api'], inplace=True)
  return df

In [ ]:
#Padronizar nome de paises
@task
def padronizar_paises(df):
    logger.info("Iniciando padronização de países...")

    def converter(country):
        try:
            if country == "Não Informado":
                return country

            #Pegar linhas com mais de 1 país
            paises = [p.strip() for p in country.split(",")]

            siglas = []

            for p in paises:
                if len(p) == 2:
                    codigo = pycountry.countries.get(alpha_2=p.upper())
                    if codigo:
                        siglas.append(codigo.alpha_2)
                    else:
                        siglas.append(p)
                else:
                    codigo = pycountry.countries.lookup(p)
                    siglas.append(codigo.alpha_2)

            return ", ".join(siglas)

        except:
            return country

    df["country"] = df["country"].apply(converter)

    logger.info("Padronização concluída!")
    return df

In [ ]:
# Padronizar classificação indicativa

@task
def padronizar_classificacao(df):
    logger.info("Iniciando padronização de classificação indicativa...")

    mapa_classificacao = {
        "G": "Livre",
        "TV-G": "Livre",
        "TV-Y": "Livre",
        "TV-Y7": "10",
        "TV-Y7-FV": "10",
        "PG": "10",
        "TV-PG": "10",
        "PG-13": "12",
        "TV-14": "14",
        "R": "16",
        "NC-17": "18",
        "TV-MA": "18",
        "NR": "Não Classificado",
        "UR": "Não Classificado",
        "66 min": "Não Informado",
        "74 min": "Não Informado",
        "84 min": "Não Informado"
    }

    df["rating"] = df["rating"].map(mapa_classificacao)

    logger.info("Padronização concluída!")
    return df

In [ ]:
#Conversão dos dados
@task
def conversao(df):
  logger.info("Iniciando conversão de tipos...")

  colunas_categoricas = ['rating', 'type']
  colunas_texto = ['show_id', 'title', 'director', 'cast', 'country', 'listed_in']
  df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')
  df['release_date_api'] = pd.to_datetime(df['release_date_api'], errors='coerce')
  df['release_year'] = df['release_year'].astype(int)
  df[colunas_categoricas] = df[colunas_categoricas].astype('category')
  df[colunas_texto] = df[colunas_texto].astype(str)

  logger.info("Conversão concluída!")
  return df

In [ ]:
#Derivação da coluna duração

@task
def derivacao_duracao(df):
  logger.info("Iniciando derivação de colunas...")

  df['duration_num'] = df['duration'].str.extract(r'(\d+)').astype(float)
  df['duration_minutes'] = np.where(df['type'] == 'Movie', df['duration_num'], np.nan) #Mantém NaN para facilitar cálculo de médias
  df['seasons_count'] = np.where(df['type'] == 'TV Show', df['duration_num'], np.nan)
  df.drop(columns=['duration', 'duration_num'], inplace=True)

  logger.info("Derivação concluída!")
  return df

In [ ]:
#Criar função que separa data de lançamento por estação
def mapear_estacao(mes):
    if mes in [12, 1, 2]:
        return "Verão"
    elif mes in [3, 4, 5]:
        return "Outono"
    elif mes in [6, 7, 8]:
        return "Inverno"
    elif mes in [9, 10, 11]:
        return "Primavera"
    else:
        return "Não informado"

#Aplicar função e criar coluna de estação
@task
def adicionar_estacao(df):
    logger.info("Criando coluna de estação...")

    df["estacao"] = df["release_date_api"].dt.month.apply(mapear_estacao)

    logger.info("Coluna 'estacao' criada com sucesso!")
    return df


In [ ]:
#Traduzir colunas
@task
def traduzir_colunas(df):
  logger.info("Traduzindo colunas...")
  df = df.rename(columns={
    'show_id': 'id',
    'type':'tipo',
    'title':'titulo',
    'director':'diretor',
    'cast':'elenco',
    'country':'pais',
    'date_added':'adicionado_em',
    'release_year':'ano_lancamento',
    'release_date_api':'data_lancamento',
    'rating':'classificacao',
    'duration_minutes':'duracao_em_min',
    'seasons_count':'temporadas',
    'listed_in':'genero'
})
  logger.info("Tradução concluída!")
  return df

In [ ]:
#Salvar CSV (Load)
OUTPUT_PATH = "dataset_final.csv"

import os

@task
def load_task(df):
    logger.info(f"Salvando dataframe em {OUTPUT_PATH}...")

    if os.path.exists(OUTPUT_PATH):
        logger.warning(f"O arquivo {OUTPUT_PATH} já existe e será sobrescrito.")

    df.to_csv(OUTPUT_PATH, index=False)

    logger.info(f"Arquivo salvo com sucesso! {len(df)} registros e {df.shape[1]} colunas.")

    return OUTPUT_PATH

In [ ]:
@flow
def etl_netflix():
  dados_brutos = criacao_df()
  dados_sem_duplicatas = remover_duplicatas(dados_brutos)
  dados_sem_descricao = remover_descricao(dados_sem_duplicatas)
  dados_sem_nulos = substituicao_nulos(dados_sem_descricao)
  dados_enriquecidos = enriquecer_com_api(dados_sem_nulos)
  dados_paises_padronizados = padronizar_paises(dados_enriquecidos)
  dados_classificacoes_padronizadas = padronizar_classificacao(dados_paises_padronizados)
  dados_tipados = conversao(dados_classificacoes_padronizadas)
  dados_duracao = derivacao_duracao(dados_tipados)
  dados_estacao = adicionar_estacao(dados_duracao)
  dataset_final = traduzir_colunas(dados_estacao)
  salvar = load_task(dataset_final)
  return dataset_final

if __name__ == "__main__":
    df_netflix = etl_netflix()
    display(f'-'*45)
    display(f'Número de nulos por coluna')
    display(f'-'*45)
    display(df_netflix.isnull().sum())
    display(f'-'*45)
    display(f'Informações gerais')
    display(f'-'*45)
    display(df_netflix.info())
    display(f'-'*45)
    display(f'Visualização do dataset')
    display(f'-'*45)
    display(df_netflix)

21:43:25 |   INFO    | criacao_df: Iniciando extração...
21:43:25 |   INFO    | criacao_df: Arquivo carregado com sucesso. Linhas: 8807
21:43:25 |   INFO    | remover_duplicatas: Iniciando verificação de duplicatas. Registros duplicados encontrados: 0
21:43:25 |   INFO    | remover_duplicatas: Duplicatas removidas!
21:43:25 |   INFO    | remover_descricao: Iniciando remoção da coluna 'description'
21:43:25 |   INFO    | remover_descricao: Remoção concluída!
21:43:25 |   INFO    | substituicao_nulos: Iniciando substituição de nulos. Campos nulos encontrados: 4307
21:43:25 |   INFO    | substituicao_nulos: Transformação concluída! Nulos: 0
21:43:26 |   INFO    | enriquecer_com_api: Iniciando merge com API...
21:43:26 |   INFO    | padronizar_paises: Iniciando padronização de países...
21:43:26 |   INFO    | padronizar_paises: Padronização concluída!
21:43:26 |   INFO    | padronizar_classificacao: Iniciando padronização de classificação indicativa...
21:43:26 |   INFO    | padronizar_cla

'---------------------------------------------'

'Número de nulos por coluna'

'---------------------------------------------'

,0
id,0
tipo,0
titulo,0
diretor,0
elenco,0
pais,0
adicionado_em,98
ano_lancamento,0
classificacao,4
genero,0


...

None

'---------------------------------------------'

'Visualização do dataset'

'---------------------------------------------'

,id,tipo,titulo,diretor,elenco,pais,adicionado_em,ano_lancamento,classificacao,genero,data_lancamento,duracao_em_min,temporadas,estacao
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,"Richard Johnson, Kirsten Johnson, Isla Sierck,...",US,2021-09-25,2020,12,Documentaries,2020-01-23,90.0,NaN,Verão
1,s2,TV Show,Blood & Water,Não Informado,"Ama Qamata, Khosi Ngema, Gail Mabalane, Dillon...",ZA,2021-09-24,2021,18,"International TV Shows, TV Dramas, TV Mysteries",2020-05-20,NaN,2.0,Outono
2,s3,TV Show,Ganglands,Não Informado,"Sami Bouajila, Tracy Gotoas, Salim Kéchiouche,...",FR,2021-09-24,2021,18,"Crime TV Shows, International TV Shows, TV Act...",2021-09-24,NaN,1.0,Primavera
3,s4,TV Show,Jailbirds New Orleans,Não Informado,Não Informado,US,2021-09-24,2021,18,"Docuseries, Reality TV",2021-09-24,NaN,1.0,Primavera
4,s5,TV Show,Kota Factory,Não Informado,"Jitendra Kumar, Mayur More, Ranjan Raj, Alam K...",IN,2021-09-24,2021,18,"International TV Shows, Romantic TV Shows, TV ...",2019-04-16,NaN,2.0,Outono
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8802,s8803,Movie,Zodiac,David Fincher,"Jake Gyllenhaal, Mark Ruffalo, Anthony Edwards...",US,2019-11-20,2007,16,"Cult Movies, Dramas, Thrillers",2007-03-02,158.0,NaN,Outono
8803,s8804,TV Show,Zombie Dumb,Não Informado,Não Informado,KR,2019-07-01,2018,10,"Kids' TV, Korean TV Shows, TV Comedies",2017-01-01,NaN,2.0,Verão
8804,s8805,Movie,Zombieland,Ruben Fleischer,"Jesse Eisenberg, Woody Harrelson, Emma Stone, ...",US,2019-11-01,2009,16,"Comedies, Horror Movies",2009-10-02,88.0,NaN,Primavera
8805,s8806,Movie,Zoom,Peter Hewitt,"Tim Allen, Courteney Cox, Chevy Chase, Spencer...",US,2020-01-11,2006,10,"Children & Family Movies, Comedies",2006-08-11,88.0,NaN,Inverno
